In [18]:
# 네이버 검색 결과화면에서 뉴스 제목과 링크 추출하기 
import requests

keyword = 'ai'
url = f'https://search.naver.com/search.naver?where=news&query={keyword}'

# requests로 url로 요청해서 응답을 가져옴
response = requests.get(url)

In [19]:
from bs4 import BeautifulSoup
import pandas as pd

try:
	# 요청이 성공했는지 확인
	response.raise_for_status()
	print("연결 성공!!")
	# 가져온 결과를 콘솔에 출력
	# print(response.text)

	# 가져온 결과에서 다음을 만족하는 요소들을 가져옴
	# .sds-comps-base-layout a[data-heatmap-target=".tit"]
	soup = BeautifulSoup(response.text, 'lxml')
	links = soup.select('.sds-comps-base-layout .XEtVZ4N7DI2Pv29xe7f3[data-heatmap-target=".tit"]')

	titles = []
	hrefs = []
	# 가져온 요소들을 콘솔에 출력 
	for link in links:
		# 제목과 링크(href)를 추출해서 콘솔에 출력
		title = link.text 
		href = link.get('href')
		# 제목과 링크를 제목 리스트, 링크 리스트에 각각 뒤에 추가
		titles.append(title)
		hrefs.append(href)

	# print(titles)
	# print(hrefs)
	# 판다스를 이용하여 DataFrame을 생성
	# 추가할 때 열 이름은 title과 href로 지정
	df = pd.DataFrame({
		'title' : titles,
		'href' : hrefs
	})
	
	# csv 파일로 저장
	df.to_csv(f'naver_nl_{keyword}.csv', encoding="utf-8-sig", index=False)
	print(len(df))
except Exception as e:
	print(f"예외 발생 : {e}")

연결 성공!!
10


In [37]:
# 위 예제를 이용하여 키워드가 주어졌을 때 뉴스 목록을 가져와서 데이터프레임으로 반환하는
# 함수를 만들고 테스트 하세요. 
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_naver_news_list(keyword : str, start : int = 1):
	url = f'https://search.naver.com/search.naver'
	params = {
		'where' : 'news', 
		'query' : keyword,
		'start' : start,
	}
	# 뉴스 검색 화면을 요청
	response = requests.get(url, params=params)
	try:
		# 연결 성공이 아니면 예외 발생
		response.raise_for_status()
		# Beautifulsoup 객체 생성
		soup = BeautifulSoup(response.text, 'lxml')
		# 뉴스 제목 링크들을 가져옴
		links = soup.select('.sds-comps-base-layout .XEtVZ4N7DI2Pv29xe7f3[data-heatmap-target=".tit"]')
		hrefs, titles = [], []
		# 반복문으로 제목과 링크를 추출하여 리스트에 담음
		for link in links:
			hrefs.append(link.get("href"))
			titles.append(link.text)
		return pd.DataFrame({
			'title': titles,
			'href' : hrefs,
		})
	except Exception as e:
		print(f"예외 발생 : {e}")
		return None

In [48]:
# 뉴스 리트스 DataFrame을 csv로 저장
def save_nl(df:pd.DataFrame, keyword:str, start:int = 1):
	# 데이터프레임이 없거나 비어 있으면
	if df is None or df.empty:
		return
	df.to_csv(f'naver_nl_{keyword}_{start//10 + 1}.csv', encoding="utf-8-sig", index=False)	


In [ ]:
start = 41
keyword = 'ai'
nl_df = get_naver_news_list(keyword, start)
save_nl(nl_df, keyword, start)

In [ ]:
# 반복문으로 총 50개의 ai 뉴스 목록을 가져오도록 작업을 하세요. 
# 단, 작업할 때 time.sleep(2)를 이용하여 2초의 딜레이를 줌
